# Fixed Egr1 CC Heatmap Visualization
## Proper deepTools heatmap generation for sex-biased peaks

Issues fixed:
1. Running computeMatrix separately for Male and Female samples
2. Using consistent z-scale across both heatmaps
3. Proper color maps for each sex (Blues for male, RdPu/Purples for female)
4. Removing yMax limit that was cutting off data

In [ ]:
import subprocess
import os
import pandas as pd
import numpy as np

# Set working directory
os.chdir('/scratch/rmlab/rmlab_shared3/l.llaci/Egr1_paper/testing_CC_forPaper')
print(f"Working directory: {os.getcwd()}")
print(f"\nFiles in directory:")
print("Peak files:")
!ls -lh combined_peaks.bed male_biased_overlaps.bed female_biased_overlaps.bed 2>/dev/null
print("\nBigWig files:")
!ls -lh ../cpm_bigwigs/*CPM.bw 2>/dev/null | head -10

## Step 1: Generate heatmap matrices using computeMatrix

Key parameters:
- `--referencePoint center`: Center peaks for signal visualization
- `-b 1000 -a 1000`: 1kb flanking regions on each side
- `--binSize 10`: 10bp bin size for resolution
- `-p 8`: Use 8 processors for speed

In [ ]:
# First, verify all input files exist
peak_files = ['combined_peaks.bed', 'male_biased_overlaps.bed', 'female_biased_overlaps.bed']
bw_files = [
    '../cpm_bigwigs/Male_Egr1_CPM.bw',
    '../cpm_bigwigs/Female_Egr1_CPM.bw'
]

print("Checking input files:")
for f in peak_files + bw_files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    status = "✓" if exists else "✗"
    print(f"  {status} {f}: {size:,} bytes")

In [ ]:
# Create a combined regions file with all three peak categories
# This will be used to create one matrix with regions labeled

import tempfile

# Read and combine the peak files with labels
combined_peaks = pd.read_csv('combined_peaks.bed', sep='\t', header=None, names=['chr', 'start', 'end'])
male_biased = pd.read_csv('male_biased_overlaps.bed', sep='\t', header=None, names=['chr', 'start', 'end'])
female_biased = pd.read_csv('female_biased_overlaps.bed', sep='\t', header=None, names=['chr', 'start', 'end'])

print(f"Loaded peaks:")
print(f"  Shared (combined): {len(combined_peaks)} peaks")
print(f"  Male-biased: {len(male_biased)} peaks")
print(f"  Female-biased: {len(female_biased)} peaks")

# Add a 4th column with region name for tracking
combined_peaks['name'] = 'Shared_' + (combined_peaks.index.astype(str))
male_biased['name'] = 'Male_biased_' + (male_biased.index.astype(str))
female_biased['name'] = 'Female_biased_' + (female_biased.index.astype(str))

# Save as individual bed files (format expected by computeMatrix)
combined_peaks[['chr', 'start', 'end']].to_csv('shared_peaks_for_heatmap.bed', sep='\t', index=False, header=False)
male_biased[['chr', 'start', 'end']].to_csv('male_biased_peaks_for_heatmap.bed', sep='\t', index=False, header=False)
female_biased[['chr', 'start', 'end']].to_csv('female_biased_peaks_for_heatmap.bed', sep='\t', index=False, header=False)

print(f"\nSaved individual peak files for heatmap")

In [ ]:
# Run computeMatrix for MALE sample
# Using all three region types to compare their signal in male context

cmd_male = [
    'computeMatrix', 'reference-point',
    '--referencePoint', 'center',
    '-b', '1000',
    '-a', '1000',
    '--binSize', '10',
    '-R',
    'shared_peaks_for_heatmap.bed',
    'male_biased_peaks_for_heatmap.bed',
    'female_biased_peaks_for_heatmap.bed',
    '-S',
    '../cpm_bigwigs/Male_Egr1_CPM.bw',
    '--missingDataAsZero',
    '-p', '8',
    '-o', 'Male_Egr1CC_heatmap_matrix.gz',
    '--verbose'
]

print("Running computeMatrix for MALE Egr1 CC...")
print(" ".join(cmd_male))
result = subprocess.run(cmd_male, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Return code: {result.returncode}")

In [ ]:
# Run computeMatrix for FEMALE sample

cmd_female = [
    'computeMatrix', 'reference-point',
    '--referencePoint', 'center',
    '-b', '1000',
    '-a', '1000',
    '--binSize', '10',
    '-R',
    'shared_peaks_for_heatmap.bed',
    'male_biased_peaks_for_heatmap.bed',
    'female_biased_peaks_for_heatmap.bed',
    '-S',
    '../cpm_bigwigs/Female_Egr1_CPM.bw',
    '--missingDataAsZero',
    '-p', '8',
    '-o', 'Female_Egr1CC_heatmap_matrix.gz',
    '--verbose'
]

print("Running computeMatrix for FEMALE Egr1 CC...")
print(" ".join(cmd_female))
result = subprocess.run(cmd_female, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Return code: {result.returncode}")

## Step 2: Calculate data statistics to determine appropriate scale

This ensures both heatmaps use consistent z-scale for proper comparison

In [ ]:
# Extract matrix data to determine z-scale
import gzip
import pickle

def load_matrix_stats(matrix_file):
    """Load matrix and compute statistics for each region type"""
    with gzip.open(matrix_file, 'rb') as f:
        data = pickle.load(f)
    
    # Extract matrix values
    matrix = data['matrix']
    labels = data['group_labels']
    
    print(f"Matrix shape: {matrix.shape}")
    print(f"Group labels: {labels}")
    print(f"\nData statistics (min, mean, max):")
    print(f"  Overall: {matrix.min():.3f}, {matrix.mean():.3f}, {matrix.max():.3f}")
    
    # Calculate stats per region type
    region_stats = {}
    for i, label in enumerate(labels):
        mask = data['group_boundaries'][i][0]:data['group_boundaries'][i][1]
        region_matrix = matrix[mask, :]
        region_stats[label] = {
            'min': region_matrix.min(),
            'mean': region_matrix.mean(),
            'max': region_matrix.max(),
            'q95': np.percentile(region_matrix, 95),
            'q99': np.percentile(region_matrix, 99)
        }
        print(f"  {label}: min={region_stats[label]['min']:.3f}, mean={region_stats[label]['mean']:.3f}, max={region_stats[label]['max']:.3f}")
        print(f"            95th%ile={region_stats[label]['q95']:.3f}, 99th%ile={region_stats[label]['q99']:.3f}")
    
    return region_stats

print("\n=== MALE Egr1 CC Matrix ===")
male_stats = load_matrix_stats('Male_Egr1CC_heatmap_matrix.gz')

print("\n=== FEMALE Egr1 CC Matrix ===")
female_stats = load_matrix_stats('Female_Egr1CC_heatmap_matrix.gz')

In [ ]:
# Determine optimal z-scale (use 99th percentile across all samples)
all_max_values = []

for stats in [male_stats, female_stats]:
    for region_type in stats:
        all_max_values.append(stats[region_type]['q99'])

z_max = np.max(all_max_values)
# Round up to nice number
z_max = np.ceil(z_max * 2) / 2  # Round to nearest 0.5

print(f"Recommended z-scale: 0 to {z_max}")
print(f"\nThis ensures both heatmaps use the same scale for proper comparison")

## Step 3: Generate heatmaps with proper visualization settings

In [ ]:
# Generate MALE heatmap
cmd_plot_male = [
    'plotHeatmap',
    '-m', 'Male_Egr1CC_heatmap_matrix.gz',
    '-o', 'Male_Egr1CC_heatmap_FIXED.pdf',
    '--colorMap', 'Blues',  # Use single colormap for males
    '--refPointLabel', 'center',
    '--regionsLabel', 'Shared', 'Male-biased', 'Female-biased',
    '--samplesLabel', 'Male Egr1 CC',
    '--xAxisLabel', 'distance from peak center (bp)',
    '--yAxisLabel', 'Peaks',
    '--zMin', '0',
    '--zMax', str(z_max),
    '--heatmapHeight', '14',
    '--dpi', '150',
    '--imageType', 'pdf'
]

print("Generating MALE heatmap...")
print(" ".join(cmd_plot_male))
result = subprocess.run(cmd_plot_male, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Return code: {result.returncode}")
print(f"\nOutput file: Male_Egr1CC_heatmap_FIXED.pdf")

In [ ]:
# Generate FEMALE heatmap
cmd_plot_female = [
    'plotHeatmap',
    '-m', 'Female_Egr1CC_heatmap_matrix.gz',
    '-o', 'Female_Egr1CC_heatmap_FIXED.pdf',
    '--colorMap', 'RdPu',  # Use RdPu colormap for females
    '--refPointLabel', 'center',
    '--regionsLabel', 'Shared', 'Male-biased', 'Female-biased',
    '--samplesLabel', 'Female Egr1 CC',
    '--xAxisLabel', 'distance from peak center (bp)',
    '--yAxisLabel', 'Peaks',
    '--zMin', '0',
    '--zMax', str(z_max),
    '--heatmapHeight', '14',
    '--dpi', '150',
    '--imageType', 'pdf'
]

print("Generating FEMALE heatmap...")
print(" ".join(cmd_plot_female))
result = subprocess.run(cmd_plot_female, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Return code: {result.returncode}")
print(f"\nOutput file: Female_Egr1CC_heatmap_FIXED.pdf")

## Step 4: Create additional visualizations (line plots)

In [ ]:
# Generate line plots for comparison
cmd_plot_profile_male = [
    'plotProfile',
    '-m', 'Male_Egr1CC_heatmap_matrix.gz',
    '-o', 'Male_Egr1CC_profile_FIXED.pdf',
    '--plotType', 'std',  # Standard deviation
    '--regionsLabel', 'Shared', 'Male-biased', 'Female-biased',
    '--samplesLabel', 'Male Egr1 CC',
    '--xAxisLabel', 'distance from peak center (bp)',
    '--yAxisLabel', 'Signal (CPM)',
    '--dpi', '150',
    '--imageType', 'pdf'
]

print("Generating MALE profile plot...")
result = subprocess.run(cmd_plot_profile_male, capture_output=True, text=True)
print(f"Return code: {result.returncode}")
print(f"Output file: Male_Egr1CC_profile_FIXED.pdf")

cmd_plot_profile_female = [
    'plotProfile',
    '-m', 'Female_Egr1CC_heatmap_matrix.gz',
    '-o', 'Female_Egr1CC_profile_FIXED.pdf',
    '--plotType', 'std',
    '--regionsLabel', 'Shared', 'Male-biased', 'Female-biased',
    '--samplesLabel', 'Female Egr1 CC',
    '--xAxisLabel', 'distance from peak center (bp)',
    '--yAxisLabel', 'Signal (CPM)',
    '--dpi', '150',
    '--imageType', 'pdf'
]

print("\nGenerating FEMALE profile plot...")
result = subprocess.run(cmd_plot_profile_female, capture_output=True, text=True)
print(f"Return code: {result.returncode}")
print(f"Output file: Female_Egr1CC_profile_FIXED.pdf")

## Summary of Fixes

### Original Issues:
1. **Missing signal on left male side**: Likely due to running computeMatrix with combined male+female samples
2. **Lower female peaks**: Color scale settings were clipping the signal
3. **Weird looking heatmap**: yMax parameter was artificially limiting the visualization

### Solutions Implemented:
1. **Separate matrices for each sex**: computeMatrix is now run separately for Male_Egr1_CPM.bw and Female_Egr1_CPM.bw
2. **Unified z-scale**: Both heatmaps use the same zMin (0) and zMax (calculated from 99th percentile)
3. **Better color maps**: Blues for males, RdPu for females (matches the expected output)
4. **Removed yMax limit**: Heatmap height is now controlled by --heatmapHeight instead
5. **Proper region ordering**: Shared → Male-biased → Female-biased for easy comparison
6. **Line plots included**: Provides complementary view of the peak signals

### Output Files:
- `Male_Egr1CC_heatmap_FIXED.pdf` - Heatmap showing Male Egr1 CC signal at all regions
- `Female_Egr1CC_heatmap_FIXED.pdf` - Heatmap showing Female Egr1 CC signal at all regions
- `Male_Egr1CC_profile_FIXED.pdf` - Profile plot with mean ± SD
- `Female_Egr1CC_profile_FIXED.pdf` - Profile plot with mean ± SD

In [ ]:
# Verify output files were created
print("\nGenerated output files:")
output_files = [
    'Male_Egr1CC_heatmap_FIXED.pdf',
    'Female_Egr1CC_heatmap_FIXED.pdf',
    'Male_Egr1CC_profile_FIXED.pdf',
    'Female_Egr1CC_profile_FIXED.pdf'
]

for f in output_files:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"  ✓ {f} ({size:,} bytes)")
    else:
        print(f"  ✗ {f} (not found)")